# CEG-WM HF-only threshold-fit GPU execution shard

This output-free Notebook checks out one audited Git revision, invokes the repository's server entrypoint for one frozen fit shard, and copies only its verified result or diagnostic ZIP plus receipt to Drive. It cannot approve tau, unlock confirmation data, run baselines, or create scientific claims.

In [ ]:
from google.colab import drive, files, userdata
from pathlib import Path
from hashlib import sha256
import json
import os
import shutil
import subprocess
import sys
import tempfile

drive.mount('/content/drive')

In [ ]:
REPOSITORY_URL = 'https://github.com/RICHAAARC/CEG-WM.git'
EXPECTED_REVISION = 'b957e5bd7996ef3f1ed365316fc381a424074ffb'
RUN_ID = 'hf-only-content-threshold-fit'
SHARD_INDEX = 0
CHECKOUT_ROOT = Path('/content/ceg_wm_source')
SCRATCH_ROOT = Path('/content/ceg_wm_scratch')
CACHE_ROOT = Path('/content/ceg_wm_cache')
OUTPUT_ROOT = Path('/content/ceg_wm_output')
DRIVE_ROOT = Path('/content/drive/MyDrive/CEG-WM')
DRIVE_SHARD_ROOT = DRIVE_ROOT / 'hf_only_threshold_fit_results' / EXPECTED_REVISION / RUN_ID / f'shard_{SHARD_INDEX:02d}'
environment = os.environ.copy()
environment['HF_TOKEN'] = userdata.get('HF_TOKEN')
environment['CEG_WM_ROOT_KEY'] = userdata.get('CEG_WM_ROOT_KEY')
assert environment['HF_TOKEN'] and environment['CEG_WM_ROOT_KEY']

In [ ]:
assert not CHECKOUT_ROOT.exists(), 'fresh checkout path required'
subprocess.run(['git', 'clone', '--no-checkout', REPOSITORY_URL, str(CHECKOUT_ROOT)], check=True)
subprocess.run(['git', '-C', str(CHECKOUT_ROOT), 'fetch', '--depth', '1', 'origin', EXPECTED_REVISION], check=True)
subprocess.run(['git', '-C', str(CHECKOUT_ROOT), 'checkout', '--detach', EXPECTED_REVISION], check=True)
observed_revision = subprocess.run(['git', '-C', str(CHECKOUT_ROOT), 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True).stdout.strip()
observed_status = subprocess.run(['git', '-C', str(CHECKOUT_ROOT), 'status', '--porcelain'], check=True, capture_output=True, text=True).stdout
assert observed_revision == EXPECTED_REVISION
assert observed_status == ''

In [ ]:
server_entrypoint = CHECKOUT_ROOT / 'scripts/experiment_execution/hf_only_threshold_fit_server.py'
command = [
    sys.executable, str(server_entrypoint),
    '--repository-root', str(CHECKOUT_ROOT),
    '--expected-revision', EXPECTED_REVISION,
    '--scratch-root', str(SCRATCH_ROOT),
    '--cache-root', str(CACHE_ROOT),
    '--output-root', str(OUTPUT_ROOT),
    '--run-id', RUN_ID,
    '--shard-index', str(SHARD_INDEX),
]
completed = subprocess.run(command, check=False, capture_output=True, text=True, env=environment)
if completed.stderr:
    print(completed.stderr)
receipt = json.loads(completed.stdout)
assert completed.returncode in (0, 3, 4)
assert receipt['bootstrap_exit_code'] == completed.returncode
assert receipt['committed_revision'] == EXPECTED_REVISION
assert receipt['run_id'] == RUN_ID and receipt['shard_index'] == SHARD_INDEX

In [ ]:
def file_sha256(path):
    digest = sha256()
    with Path(path).open('rb') as source:
        for block in iter(lambda: source.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def atomic_copy_verified(source, destination, expected_sha256):
    source = Path(source)
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        raise RuntimeError('Drive destination already exists')
    with tempfile.NamedTemporaryFile(dir=destination.parent, prefix=f'.{destination.name}.', suffix='.tmp', delete=False) as handle:
        temporary = Path(handle.name)
        with source.open('rb') as input_stream:
            shutil.copyfileobj(input_stream, handle)
        handle.flush()
        os.fsync(handle.fileno())
    try:
        if file_sha256(temporary) != expected_sha256:
            raise RuntimeError('copied file SHA-256 mismatch')
        temporary.replace(destination)
        if file_sha256(destination) != expected_sha256:
            raise RuntimeError('Drive file SHA-256 mismatch')
    finally:
        if temporary.exists():
            temporary.unlink()
    return destination

artifact_source = Path(receipt['artifact_path'])
receipt_source = Path(receipt['receipt_path'])
assert file_sha256(artifact_source) == receipt['artifact_sha256']
artifact_destination = atomic_copy_verified(artifact_source, DRIVE_SHARD_ROOT / artifact_source.name, receipt['artifact_sha256'])
receipt_sha256 = file_sha256(receipt_source)
receipt_destination = atomic_copy_verified(receipt_source, DRIVE_SHARD_ROOT / f'{artifact_source.stem}.execution_receipt.json', receipt_sha256)
transfer = {
    'artifact_kind': receipt['artifact_kind'],
    'artifact_path': str(artifact_destination),
    'artifact_sha256': file_sha256(artifact_destination),
    'receipt_path': str(receipt_destination),
    'receipt_sha256': file_sha256(receipt_destination),
    'committed_revision': EXPECTED_REVISION,
    'run_id': RUN_ID,
    'shard_index': SHARD_INDEX,
}
print(json.dumps(transfer, indent=2, sort_keys=True))
files.download(str(artifact_destination))